# 🎓 Workshop: Building an Autonomous Research Agent with LangChain & Groq

In this hands-on workshop, we will build an autonomous **Research Agent** using:
- **LangChain & LangGraph**: Agent orchestration via the ReAct (Reasoning + Acting) framework
- **ChatGroq**: Ultra-fast LLM inference using open-weight models (e.g., `openai/gpt-oss-120b`)
- **Research Tools (3 Max)**: Web search (DuckDuckGo), Academic search (ArXiv), and Background knowledge (Wikipedia)
- **System Prompting**: Designing an expert Senior Researcher persona


## 📦 Step 1: Install Required Libraries
Run the cell below to install LangChain, LangGraph, Groq, and the tool dependencies.

In [1]:
# !pip install -q langchain-tavily
!pip install -q -U langchain langchain-groq langgraph wikipedia "arxiv<4" langchain-exa firecrawl-py pydantic

In [2]:
# NOTE: LangChain's DuckDuckGo tool now needs the `ddgs` package (renamed from `duckduckgo-search`),
# and arxiv 4.x removed `Search.results()` which LangChain's ArxivAPIWrapper still calls -> pin arxiv<4.
# !pip install -q -U langchain langchain-groq langchain-community langgraph ddgs wikipedia "arxiv<4"

In [ ]:
import os
import getpass
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_exa import ExaSearchResults
from langchain_community.document_loaders import FireCrawlLoader
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph.errors import GraphRecursionError
from dotenv import load_dotenv

load_dotenv()

/tmp/ipykernel_22569/253602278.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun


## 🔑 Step 2: Set Up Groq API Key
Enter your Groq API key below. You can obtain a free key from [console.groq.com](https://console.groq.com).

In [ ]:
import os
import getpass

os.environ["GROQ_API_KEY"] 
os.environ["EXA_API_KEY"] 
os.environ["FIRECRAWL_API_KEY"]



In [5]:
# os.environ["TAVILY_API_KEY"] = "tvly-24Cea8-K2Qk9B5u6LX6kDYRPj4Dwq1MzxETL16SI17gWLB2Aa"

## 🤖 Step 3: Initialize the LLM (ChatGroq)
We initialize `ChatGroq` with an open-weight model that supports tool calling.

> ⚠️ `llama-3.3-70b-versatile` was **shut down on Groq for free/developer keys on 16 Aug 2026**. We use `openai/gpt-oss-120b` instead. If it ever gets retired too, check [console.groq.com/docs/deprecations](https://console.groq.com/docs/deprecations) and change `MODEL_NAME` only.

In [6]:
from langchain_groq import ChatGroq

MODEL_NAME = "openai/gpt-oss-120b"   # change here only if Groq retires this model

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    max_tokens=4096,   # 2048 truncated the structured report (reasoning tokens count against this too)
    reasoning_effort="low"
)

print(f"✅ LLM initialized with model: {llm.model_name}")

✅ LLM initialized with model: openai/gpt-oss-120b


##  Define Tools (Exa, Firecrawl, ArXiv, Wiki)

In [7]:
# 3.a. Exa (Semantic Web Search)
exa_tool = ExaSearchResults(name="exa_search", max_results=2)
# 3.b. Firecrawl (Deep Web Scraping)
@tool
def firecrawl_scrape(url: str) -> str:
    """Scrapes the full markdown content of a provided URL. Use this to read the content of links found by exa_search."""
    try:
        loader = FireCrawlLoader(api_key=os.environ["FIRECRAWL_API_KEY"], url=url, mode="scrape")
        docs = loader.load()
        # Limit to 1500 characters to avoid blowing up the LLM context window
        return docs[0].page_content[:400]
    except Exception as e:
        return f"Error scraping {url}: {str(e)}"

# 3.c. ArXiv & Wikipedia (Academic & Background)
arxiv_wrapper = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper, name="arxiv_academic_search")

wiki_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper, name="wikipedia_search")

tools = [exa_tool, firecrawl_scrape, arxiv_tool, wiki_tool]
print(f"✅ {len(tools)} Tools configured: {[t.name for t in tools]}")

✅ 4 Tools configured: ['exa_search', 'firecrawl_scrape', 'arxiv_academic_search', 'wikipedia_search']


##  Define Query Expansion Logic

In [8]:
class ExpandedQueries(BaseModel):
    queries: list[str] = Field(description="List of 2 highly targeted search engine queries")

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert research planner. Take the user's topic and generate 2 highly targeted search engine queries to cover different aspects of the topic. Return ONLY the queries in the structured format."),
    ("human", "{topic}")
])

query_expander = rewrite_prompt | llm.with_structured_output(ExpandedQueries)

## 🧠 Step 5: Define the Researcher System Prompt
The system prompt instructs the agent on role adherence, tool selection rules, fact verification, and structured report synthesis.

Query Exansion

In [9]:
SYSTEM_PROMPT = """You are an expert, meticulous Senior Research Agent.
Your goal is to conduct comprehensive, objective, and well-supported research to answer the user's questions.

Follow this systematic research methodology:
1. **Tool Selection Strategy**:
  - Use `exa_search` for state-of-the-art semantic web search and finding relevant URLs.
  - Use `firecrawl_scrape` on the URLs found by Exa to read their actual content. Do NOT hallucinate content.
  - Use `wikipedia_search` for core definitions and overview knowledge.
  - Use `arxiv_academic_search` for formal scientific studies and algorithms.
2. **Verify & Synthesize**: Compare findings. Attribute claims to the retrieved tool data.
3. **Structured Delivery**: Present your final report clearly with:
  - **Executive Summary**
  - **Key Findings & Analysis**
  - **Supporting Evidence**
  - **References (Links & Sources)**
"""

print("✅ System prompt configured!")

✅ System prompt configured!


## ⚡ Step 6: Create the ReAct Agent (`create_react_agent`)
We use `create_react_agent` from LangGraph to connect the LLM, the 3 tools, and the researcher prompt into an autonomous loop.

We wrap the tools in a `ToolNode` with `handle_tool_errors=True`: by default, a tool that raises (DuckDuckGo rate-limit, Wikipedia timeout, network error…) **crashes the whole run**. With this setting the error is sent back to the model as a message, so it can retry or switch tools.

In [10]:
tool_node = ToolNode(tools, handle_tool_errors=True)
research_agent = create_react_agent(model=llm, tools=tool_node, prompt=SYSTEM_PROMPT)
print("✅ Research Agent with LangGraph built successfully!")

✅ Research Agent with LangGraph built successfully!


/tmp/ipykernel_22569/4035507475.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  research_agent = create_react_agent(model=llm, tools=tool_node, prompt=SYSTEM_PROMPT)


## 🔬 Step 7: Define Execution & Streaming Function
This helper function sends a research query to the agent and streams its reasoning, tool invocations, and final report in real-time.

In [11]:
MAX_STEPS = 40

def run_advanced_research(user_topic: str):
    print("\n" + "=" * 70)
    print(f"🔬 INITIAL TOPIC: {user_topic}")
    print("=" * 70)

    # Step A: Expand Queries
    print("🔄 Expanding Queries to enhance search quality...")
    try:
        expanded = query_expander.invoke({"topic": user_topic})
        print("🎯 Generated Queries:")
        for i, q in enumerate(expanded.queries, 1):
            print(f"   {i}. {q}")
    except Exception as e:
        print(f"⚠️ Query expansion failed, proceeding with original topic. Error: {e}")
        expanded = ExpandedQueries(queries=[user_topic])

    # Step B: Prepare combined instructions
    combined_instructions = (
        f"The user wants to research: '{user_topic}'.\n"
        f"I have expanded this into specific queries for you to investigate:\n"
        f"{chr(10).join(['- ' + q for q in expanded.queries])}\n\n"
        "Execute searches, scrape important links with Firecrawl if needed, and synthesize a final markdown report."
    )

    inputs = {"messages": [HumanMessage(content=combined_instructions)]}
    config = {"recursion_limit": MAX_STEPS}
    final_response = ""
    seen = 0

    print("\n🚀 Starting Agent Execution (Reasoning & Acting)...")
    try:
        for chunk in research_agent.stream(inputs, config=config, stream_mode="values"):
            new_messages = chunk["messages"][seen:]
            seen = len(chunk["messages"])
            for msg in new_messages:
                if msg.type == "ai":
                    for tc in msg.tool_calls:
                        print(f"🛠️  [Action] Calling: `{tc['name']}` -> {tc['args']}")
                    if msg.content and not msg.tool_calls:
                        final_response = msg.content
                elif msg.type == "tool":
                    status = "❌ error" if getattr(msg, "status", "") == "error" else f"{len(msg.content)} chars received"
                    print(f"📥 [Result: {msg.name}] ({status})")
    except GraphRecursionError:
        print(f"\n⚠️ Agent hit the {MAX_STEPS}-step limit before finishing.")

    print("\n" + "=" * 70)
    print("📋 FINAL RESEARCH REPORT")
    print("=" * 70)
    print(final_response if final_response else "(No final report produced.)")

    return final_response

## 🚀 Step 8: Live Demonstration / Research Queries
Test the agent on different research topics and observe how it autonomously selects appropriate tools (Wikipedia vs. ArXiv vs. Web Search).

In [13]:
# Test Query 1: Academic / Cutting-Edge AI Topic
topic = "What are recent advances in Test-Time Compute and Reasoning Models (e.g. DeepSeek-R1, OpenAI o-series)?"
report = run_advanced_research(topic)


🔬 INITIAL TOPIC: What are recent advances in Test-Time Compute and Reasoning Models (e.g. DeepSeek-R1, OpenAI o-series)?
🔄 Expanding Queries to enhance search quality...
🎯 Generated Queries:
   1. latest research papers and benchmark results on Test-Time Compute and Reasoning Models such as DeepSeek-R1 and OpenAI o-series 2023-2024
   2. comparative analysis of architecture, training methods, and inference efficiency in Test-Time Compute models like DeepSeek-R1 versus OpenAI o-series

🚀 Starting Agent Execution (Reasoning & Acting)...
🛠️  [Action] Calling: `exa_search` -> {'num_results': 5, 'query': 'DeepSeek-R1 test-time compute reasoning model 2023', 'summary': True}
📥 [Result: exa_search] (9626 chars received)
🛠️  [Action] Calling: `exa_search` -> {'num_results': 5, 'query': 'OpenAI o1 series test-time compute reasoning 2024 benchmark', 'summary': True}
📥 [Result: exa_search] (9044 chars received)
🛠️  [Action] Calling: `firecrawl_scrape` -> {'url': 'https://openai.com/index/introdu

In [ ]:
# # Test Query 2: Interactive / Custom Topic
# custom_topic = input("Enter any research topic to test: ") or "Quantum Error Correction breakthroughs in 2025"
# report = run_advanced_research(custom_topic)